## Interactive One-Stage Convolutional Network

This notebook demonstrates a single-stage convolutional network where you can interactively choose an input image, select different convolution kernels, and adjust parameters like kernel size, dilation, stride, and padding. The output of the convolution will be displayed live.

In [1]:
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Ensure TensorFlow uses eager execution for easier debugging if needed
tf.config.run_functions_eagerly(True)

print("Libraries imported successfully.")

Libraries imported successfully.


### 1. Image Selection

Choose one of the provided sample JPEG images to apply the convolution to.

In [2]:
image_files = ['/content/Sample 1.jpeg', '/content/Sample 2.jpeg']

# Check if the sample files exist
for img_file in image_files:
    if not os.path.exists(img_file):
        print(f"Warning: {img_file} not found. Please ensure it's in the /content/ directory.")

image_selector = widgets.Dropdown(
    options=image_files,
    value=image_files[0] if image_files else None,
    description='Select Image:'
)

# Global variable to store the selected image data
selected_image_data = None

def load_selected_image(change):
    global selected_image_data
    image_path = change.new
    try:
        img = Image.open(image_path).convert('L') # Convert to grayscale
        selected_image_data = np.array(img, dtype=np.float32) / 255.0 # Normalize to [0, 1]
        print(f"Loaded image: {image_path}, Shape: {selected_image_data.shape}")
    except Exception as e:
        selected_image_data = None
        print(f"Error loading image {image_path}: {e}")

image_selector.observe(load_selected_image, names='value')

# Load initial image
if image_selector.value:
    img = Image.open(image_selector.value).convert('L')
    selected_image_data = np.array(img, dtype=np.float32) / 255.0

display(image_selector)

Dropdown(description='Select Image:', options=('/content/Sample 1.jpeg', '/content/Sample 2.jpeg'), value='/co…

### 2. Define Convolution Kernels

Here are some pre-defined 3x3 kernels for various image processing effects.

In [3]:
kernels = {
    'Identity': np.array([[0, 0, 0],
                          [0, 1, 0],
                          [0, 0, 0]], dtype=np.float32),
    'Edge Detection (Laplacian)': np.array([[-1, -1, -1],
                                            [-1,  8, -1],
                                            [-1, -1, -1]], dtype=np.float32),
    'Sharpen': np.array([[ 0, -1,  0],
                         [-1,  5, -1],
                         [ 0, -1,  0]], dtype=np.float32),
    'Blur (Box)': np.array([[1/9, 1/9, 1/9],
                            [1/9, 1/9, 1/9],
                            [1/9, 1/9, 1/9]], dtype=np.float32),
    'Emboss': np.array([[-2, -1,  0],
                        [-1,  1,  1],
                        [ 0,  1,  2]], dtype=np.float32)
}

kernel_types = list(kernels.keys())

print("Convolution kernels defined.")

Convolution kernels defined.


### 3. Convolution Function and Interactive Controls

Use the widgets below to select a kernel and adjust convolution parameters. The original and convolved images will be displayed.

In [6]:
def apply_convolution(image_data, kernel_type, kernel_size, dilations, strides, padding_size):
    if image_data is None:
        print("No image selected or image failed to load.")
        return

    # Reshape image for TensorFlow: [batch, height, width, channels]
    image_tensor = tf.constant(image_data[tf.newaxis, :, :, tf.newaxis], dtype=tf.float32)

    # Get the selected kernel
    kernel = kernels[kernel_type]

    # Adjust kernel to desired size, if different from 3x3
    if kernel_size != 3:
        # Simple resizing by interpolation or padding/cropping
        # For simplicity, we'll just use the 3x3 kernel and scale it if size is changed
        # A more complex approach would involve generating a new kernel based on type and type
        print(f"Warning: Kernel size {kernel_size} is selected, but predefined kernels are 3x3. Using 3x3 kernel.")
        # If you wanted to dynamically create kernels of different sizes, this would be the place.
        # For this example, we'll stick to the 3x3 predefined ones and ignore kernel_size slider for kernel shape.

    # Reshape kernel for TensorFlow: [height, width, in_channels, out_channels]
    # For grayscale images, in_channels = 1, out_channels = 1
    kernel_tensor = tf.constant(kernel[:, :, tf.newaxis, tf.newaxis], dtype=tf.float32)

    # Padding configuration
    # 'VALID' means no padding, 'SAME' means padding to ensure output size is same as input (stride=1)
    # For explicit padding_size, we need to manually pad the image tensor
    if padding_size > 0:
        # Pad the image manually
        paddings = tf.constant([[0, 0], [padding_size, padding_size], [padding_size, padding_size], [0, 0]])
        image_tensor = tf.pad(image_tensor, paddings, "CONSTANT")
        padding_mode = 'VALID' # After manual padding, use VALID convolution
    else:
        padding_mode = 'VALID'

    # Apply convolution
    try:
        output_tensor = tf.nn.conv2d(
            input=image_tensor,
            filters=kernel_tensor,
            strides=[1, strides, strides, 1], # strides in height and width
            padding=padding_mode,
            dilations=[1, dilations, dilations, 1] # dilations in height and width
        )
        output_image_data = output_tensor[0, :, :, 0].numpy() # Remove batch and channel dims
    except Exception as e:
        print(f"Error during convolution: {e}")
        return

    # Display results
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title('Original Image')
    plt.imshow(image_data, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(f'Convolved Output ({kernel_type})')
    plt.imshow(output_image_data, cmap='gray')
    plt.axis('off')
    plt.show()


# Create widgets for interaction
kernel_type_widget = widgets.Dropdown(
    options=kernel_types,
    value='Identity',
    description='Kernel Type:'
)

# Note: For simplicity, kernel_size currently doesn't change the shape of predefined 3x3 kernels,
# but it could be used to dynamically generate kernels of different sizes.
kernel_size_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=9,
    step=2,
    description='Kernel Size (N x N):'
)

dilations_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description='Dilations:'
)

strides_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description='Strides:'
)

padding_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=5,
    step=1,
    description='Padding Size:'
)

# Link widgets to the convolution function
def interactive_convolution(kernel_type, kernel_size, dilations, strides, padding_size):
    clear_output(wait=True)
    # Call the convolution function with the currently selected image and widget values
    apply_convolution(selected_image_data, kernel_type, kernel_size, dilations, strides, padding_size)


# Combine widgets into an interactive view
ui = widgets.VBox([
    image_selector, # Added image_selector to the UI for consistent display
    kernel_type_widget,
    kernel_size_widget,
    dilations_widget,
    strides_widget,
    padding_widget
])

out = widgets.interactive_output(interactive_convolution, {
    'kernel_type': kernel_type_widget,
    'kernel_size': kernel_size_widget,
    'dilations': dilations_widget,
    'strides': strides_widget,
    'padding_size': padding_widget
})

# Display the UI and output
display(ui, out)

Output()

### 4. Visualizing Convolution Mathematics (Focus on Strides)

Let's break down the mathematical operation of convolution, particularly focusing on how the `stride` parameter affects the output. Convolution involves sliding a small matrix (the kernel or filter) over the input image and computing the dot product at each position.

**Key Concepts:**
*   **Kernel/Filter:** The small matrix that slides over the input.
*   **Input Image:** The data we are convolving.
*   **Stride:** The number of pixels the kernel shifts over the input at each step. A stride of 1 means moving one pixel at a time, while a stride of 2 means moving two pixels at a time, effectively downsampling the output.
*   **Padding:** Adding extra pixels (usually zeros) around the border of the input to control the output size.

Let's consider a small example with a 3x3 input and a 2x2 kernel.

In [7]:
import numpy as np

# Small example input image (grayscale)
example_input = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
], dtype=np.float32)

# A simple 2x2 kernel
example_kernel = np.array([
    [1, 0],
    [0, 1]
], dtype=np.float32)

print("Example Input:")
print(example_input)
print("\nExample Kernel:")
print(example_kernel)

Example Input:
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]

Example Kernel:
[[1. 0.]
 [0. 1.]]


#### Convolution with Stride = 1

With a stride of 1, the kernel moves one pixel at a time both horizontally and vertically. Let's trace the steps:

**Input:**
```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

**Kernel:**
```
[[1, 0],
 [0, 1]]
```

**Step 1: Top-left corner (0,0)**

Kernel over `[[1, 2], [4, 5]]`

Calculation: `(1*1) + (2*0) + (4*0) + (5*1) = 1 + 0 + 0 + 5 = 6`

**Step 2: Top-middle (0,1)**

Kernel over `[[2, 3], [5, 6]]`

Calculation: `(2*1) + (3*0) + (5*0) + (6*1) = 2 + 0 + 0 + 6 = 8`

**Step 3: Middle-left (1,0)**

Kernel over `[[4, 5], [7, 8]]`

Calculation: `(4*1) + (5*0) + (7*0) + (8*1) = 4 + 0 + 0 + 8 = 12`

**Step 4: Middle-right (1,1)**

Kernel over `[[5, 6], [8, 9]]`

Calculation: `(5*1) + (6*0) + (8*0) + (9*1) = 5 + 0 + 0 + 9 = 14`

**Output for Stride = 1 (VALID padding):**
```
[[ 6,  8],
 [12, 14]]
```

In [8]:
import tensorflow as tf

# Reshape input and kernel for TensorFlow conv2d
input_tensor = tf.constant(example_input[tf.newaxis, :, :, tf.newaxis], dtype=tf.float32)
kernel_tensor = tf.constant(example_kernel[:, :, tf.newaxis, tf.newaxis], dtype=tf.float32)

# Apply convolution with stride = 1
output_stride_1 = tf.nn.conv2d(
    input=input_tensor,
    filters=kernel_tensor,
    strides=[1, 1, 1, 1], # Stride 1 in height and width
    padding='VALID'
)

print("TensorFlow Output (Stride = 1):")
print(output_stride_1[0, :, :, 0].numpy())

TensorFlow Output (Stride = 1):
[[ 6.  8.]
 [12. 14.]]


#### Convolution with Stride = 2

With a stride of 2, the kernel moves two pixels at a time both horizontally and vertically. This means it skips positions, leading to a smaller output. The calculation for each step remains the same, but the starting positions of the kernel are further apart.

**Input:**
```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

**Kernel:**
```
[[1, 0],
 [0, 1]]
```

**Step 1: Top-left corner (0,0)**

Kernel over `[[1, 2], [4, 5]]`

Calculation: `(1*1) + (2*0) + (4*0) + (5*1) = 1 + 0 + 0 + 5 = 6`

Since the stride is 2, the next horizontal position the kernel starts at would be `(0, 2)` (index 2). For a 2x2 kernel, this would go out of bounds of the 3x3 input. Similarly, the next vertical position would be `(2, 0)`, which also goes out of bounds for a 2x2 kernel starting there.

**Output for Stride = 2 (VALID padding):**
```
[[ 6]]
```

As you can see, increasing the stride reduces the size of the output feature map, as the kernel takes fewer steps across the input.

In [10]:
import tensorflow as tf

# Reshape input and kernel for TensorFlow conv2d
input_tensor = tf.constant(example_input[tf.newaxis, :, :, tf.newaxis], dtype=tf.float32)
kernel_tensor = tf.constant(example_kernel[:, :, tf.newaxis, tf.newaxis], dtype=tf.float32)

# Apply convolution with stride = 2
output_stride_2 = tf.nn.conv2d(
    input=input_tensor,
    filters=kernel_tensor,
    strides=[1, 2, 2, 1], # Stride 2 in height and width
    padding='VALID'
)

print("TensorFlow Output (Stride = 2):")
print(output_stride_2[0, :, :, 0].numpy())

TensorFlow Output (Stride = 2):
[[6.]]
